In [1]:
import numpy as np
import pandas as pd
import cv2
import pytesseract
from glob import glob
import spacy
import re 
import string

In [2]:
def cleanText(txt):
    whitespace = string.whitespace
    punctuation = "!#$%&\'()*+:;<=>?[\\]^`{|}~"
    tableWhitespace = str.maketrans('','',whitespace)
    tablePunctuation = str.maketrans('','',punctuation)
    text = str(txt)
    text = text.lower()
    removewhitespace = text.translate(tableWhitespace)
    removepunctuation = removewhitespace.translate(tablePunctuation)
    
    return str(removepunctuation)
    

In [3]:
### Load NER model
model_ner = spacy.load('./output/model-best/')

D:\Project\Business Card Reader Web App\docuscan\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_pipeline' (0.0.0) was trained with spaCy v3.1.3 and may not be 100% compatible with the current version (3.8.16). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [27]:
image = cv2.imread('./data/7.jpg')

# extract data using Pytesseract 
tessData = pytesseract.image_to_data(image)
# convert into dataframe
tessList = list(map(lambda x:x.split('\t'), tessData.split('\n')))
df = pd.DataFrame(tessList[1:],columns=tessList[0])
df.dropna(inplace=True) # drop missing values
df['text'] = df['text'].apply(cleanText)

# convet data into content
df_clean = df.query('text != "" ')
content = " ".join([w for w in df_clean['text']])
print(content)
# get prediction from NER model
doc = model_ner(content)

In [28]:
from spacy import displacy

In [29]:
displacy.render(doc, style='ent', jupyter=True)

D:\Project\Business Card Reader Web App\docuscan\Lib\site-packages\spacy\displacy\__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [30]:
docjson = doc.to_json()
docjson.keys()


dict_keys(['text', 'ents', 'sents', 'tokens'])

In [31]:
doc_text = docjson['text']

In [32]:
datafram_tokens = pd.DataFrame(docjson['tokens'])

datafram_tokens['token'] = datafram_tokens[['start', 'end']].apply(
    lambda x: doc_text[x.iloc[0]:x.iloc[1]], axis=1
)

datafram_tokens.head(10)

KeyError: "None of [Index(['start', 'end'], dtype='str')] are in the [columns]"

In [ ]:
right_table = pd.DataFrame(docjson['ents'])[['start','label']]
datafram_tokens = pd.merge(datafram_tokens,right_table,how='left',on='start')

In [33]:
datafram_tokens.fillna('O',inplace=True)
datafram_tokens.head(10)

""


In [34]:
df_clean['end'] = df_clean['text'].apply(lambda x: len(x) + 1).cumsum() - 1

df_clean['start'] = df_clean[['text', 'end']].apply(
    lambda x: x.iloc[1] - len(x.iloc[0]), axis=1
)

ValueError: Cannot set a DataFrame with multiple columns to the single column start

In [35]:
# inner join with start 
dataframe_info = pd.merge(df_clean,datafram_tokens[['start','token','label']],how='inner',on='start')

KeyError: "None of [Index(['start', 'token', 'label'], dtype='str')] are in the [columns]"

In [36]:
dataframe_info.tail(10)

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text,end,start,token,label
39,5,1,6,3,1,6,495,474,53,21,95.750923,018.,305,301,018,O
40,5,1,6,3,2,1,46,506,430,27,86.369003,"lictsrikant8099948528.blogspot.in,",340,306,lictsrikant8099948528.blogspot.in,I-WEB
41,5,1,6,3,2,2,488,506,429,27,86.235695,interviewsinhyderabad.blogspot.in,374,341,interviewsinhyderabad.blogspot.in,B-WEB
42,5,1,6,3,3,1,44,539,472,25,89.668602,"facebook.com/lictsrikant8099948528,",410,375,facebook.com/lictsrikant8099948528,O
43,5,1,6,3,3,2,526,539,443,22,90.858559,facebook.com/thathineni.srikanth.9,445,411,facebook.com/thathineni.srikanth.9,O
44,5,1,6,3,4,1,46,571,106,21,96.051147,promote,453,446,promote,O
45,5,1,6,3,4,2,161,576,56,22,96.051147,your,458,454,your,O
46,5,1,6,3,4,3,226,571,111,21,96.122276,business,467,459,business,O
47,5,1,6,3,4,4,347,571,74,21,92.602463,online,474,468,online,O
48,5,1,6,3,4,5,432,571,96,27,92.419670,pybo,479,475,pybo,O


In [37]:
bb_df = dataframe_info.query("label != 'O' ")
img = image.copy()

for x,y,w,h,label in bb_df[['left','top','width','height','label']].values:
    x = int(x)
    y = int(y)
    w = int(w)
    h = int(h)
    
    cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.putText(img,str(label),(x,y),cv2.FONT_HERSHEY_PLAIN,1,(255,0,255),2)
    
    
cv2.imshow('Predictions',img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [45]:
bb_df['label'] = bb_df['label'].apply(lambda x: x[2:])
bb_df.head()

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text,end,start,token,label,group,right,bottom
1,5,1,3,1,1,3,822,53,203,28,96.741409,8099948528,15,5,8099948528,ONE,1,1025,81
3,5,1,3,2,1,2,822,95,203,28,94.959991,8466045457,29,19,8466045457,ONE,1,1025,123
5,5,1,3,2,2,3,709,136,316,31,88.490730,lictsrikant@gmail.com,57,36,lictsrikant@gmail.com,AIL,2,1025,167
6,5,1,3,3,1,1,46,170,33,14,96.568634,life,62,58,life,G,3,79,184
7,5,1,3,3,1,2,85,151,92,42,94.741684,insurance,72,63,insurance,G,3,177,193


In [46]:
# group the label
class groupgen():
    def __init__(self):
        self.id = 0
        self.text = ''
        
    def getgroup(self,text):
        if self.text == text:
            return self.id
        else:
            self.id +=1
            self.text = text
            return self.id
        
grp_gen = groupgen()

In [47]:
bb_df['group'] = bb_df['label'].apply(grp_gen.getgroup)

In [48]:
# right and bottom of bounding box
bb_df[['left','top','width','height']] = bb_df[['left','top','width','height']].astype(int)
bb_df['right'] = bb_df['left'] + bb_df['width']
bb_df['bottom'] = bb_df['top'] + bb_df['height']

In [49]:
# tagging: groupby group
col_group = ['left','top','right','bottom','label','token','group']
group_tag_img = bb_df[col_group].groupby(by='group')

In [50]:
img_tagging = group_tag_img.agg({
    'left': 'min',
    'right': 'max',
    'top': 'min',
    'bottom': 'max',
    'label': lambda x: np.unique(x).tolist(),
    'token': lambda x: " ".join(x.astype(str))
})

In [51]:
img_tagging

,left,right,top,bottom,label,token
group,,,,,,
1,822,1025,53,123,[ONE],8099948528 8466045457
2,709,1025,136,167,[AIL],lictsrikant@gmail.com
3,46,374,151,193,[G],life insurance corporation of india
4,668,1025,163,209,[AIL],seosrikantht@gmail.com
5,310,755,227,259,[ME],thathineni srikanth
6,399,669,271,296,[S],insurance advisor
7,47,882,395,427,[G],life insurance corporation of india
8,46,917,506,533,[B],lictsrikant8099948528.blogspot.in interviewsin...


In [52]:
img_bb = image.copy()

for l, r, t, b, label, token in img_tagging.values:
    cv2.rectangle(img_bb, (l, t), (r, b), (0, 255, 0), 2)

    cv2.putText(
        img_bb, str(label), (l, t),
        cv2.FONT_HERSHEY_PLAIN, 1, (255, 0, 255), 2
    )

cv2.imshow('Bounding Box BusinessCard', img_bb)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [53]:
def parser(text,label):
    if label == 'PHONE':
        text = text.lower()
        text = re.sub(r'\D','',text)
        
    elif label == 'EMAIL':
        text = text.lower()
        allow_special_char = '@_.'
        text = re.sub(r'[^A-Za-z0-9{} ]'.format(allow_special_char),'',text)
        
    elif label == 'WEB':
        text = text.lower()
        allow_special_char = ':/.%#'
        text = re.sub(r'[^A-Za-z0-9{} ]'.format(allow_special_char),'',text)
        
    elif label in ('NAME', 'DES'):
        text = text.lower()
        text = re.sub(r'[^a-z ]','',text)
        text = text.title()
        
    elif label == 'ORG':
        text = text.lower()
        text = re.sub(r'[^a-z0-9 ]','',text)
        text = text.title()
        
    return text

In [54]:
info_array = dataframe_info[['token','label']].values
entities = dict(NAME=[],ORG=[],DES=[],PHONE=[],EMAIL=[],WEB=[])
previous = 'O'

for token, label in info_array:
    bio_tag = label[0]
    label_tag = label[2:]
    
    # step -1 parse the token
    text = parser(token,label_tag)
    
    if bio_tag in ('B','I'):
        
        if previous != label_tag:
            entities[label_tag].append(text)
            
        else:
            if bio_tag == "B":
                entities[label_tag].append(text)
                
            else:
                if label_tag in ("NAME",'ORG','DES'):
                    entities[label_tag][-1] = entities[label_tag][-1] + " " + text
                    
                else:
                    entities[label_tag][-1] = entities[label_tag][-1] + text
                    
    
    
    previous = label_tag


In [56]:
entities

{'NAME': ['Thathineni Srikanth'],
 'ORG': ['Life Insurance Corporation Of India',
  'Life Insurance Corporation Of India'],
 'DES': ['Insurance Advisor'],
 'PHONE': ['8099948528', '8466045457'],
 'EMAIL': ['lictsrikant@gmail.com', 'seosrikantht@gmail.com'],
 'WEB': ['lictsrikant8099948528.blogspot.in',
  'interviewsinhyderabad.blogspot.in']}